# P4 — Orquestación con Prefect

**RA/CE**: RA4a (estrategias corporativas basadas en datos)
**Fase**: F4 — Orquestación con Prefect
**Teoría asociada**: `01-teoria/05_orquestacion_prefect.md`

---

## Objetivos de Aprendizaje

1. Definir flujos Prefect con `@flow` y tareas con `@task`
2. Orquestar un pipeline ML completo: datos → entrenamiento → registro
3. Implementar reintentos y recuperación ante fallos
4. Conectar Prefect con MLflow Tracking
5. Evaluar cómo la orquestación automatiza decisiones (RA4a)

## Prerrequisitos

- [ ] Concepto de orquestación (visto en UD5)
- [ ] Finalizadas las prácticas P2 (pipeline datos) y P3 (MLflow)
- [ ] Prefect instalado: `pip install prefect`
- [ ] MLflow instalado: `pip install mlflow`
- [ ] Datos de ejemplo en `02-ejemplos/pipeline_completo/data/`

## Contexto

En P2 construiste un pipeline de datos y en P3 registraste experimentos con MLflow.
Ambos dependían de que tú ejecutaras los scripts manualmente. Ahora vamos a
**orquestar ese flujo completo con Prefect**: desde la carga de datos hasta el
registro del modelo, con reintentos automáticos y trazabilidad.

---
## Parte 1: Verifica tu entorno

In [ ]:
import sys, os, json

HAS_DEPS = True
missing = []

try:
    from prefect import flow, task
    from prefect.logging import get_run_logger
    print('✅ Prefect OK')
except ImportError as e:
    HAS_DEPS = False
    missing.append('prefect')
    print(f'❌ Missing: prefect')

try:
    import mlflow
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, f1_score
    print('✅ ML dependencies OK')
except ImportError as e:
    HAS_DEPS = False
    mod = str(e).split("'")[1] if "'" in str(e) else str(e)
    missing.append(mod)
    print(f'❌ Missing: {mod}')

try:
    import pandera as pa
    print('✅ Pandera OK')
except ImportError:
    print('⚠️ Pandera not installed. Optional for validation.')

if not HAS_DEPS:
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\n✅ Environment ready')
    print(f'Python: {sys.version[:6]}')

---
## Parte 2: Crear un flujo Prefect básico

Vamos a definir un flujo Prefect con tres tareas: cargar datos, limpiarlos y
entrenar un modelo. Observa cómo los decoradores `@task` y `@flow` convierten
funciones Python normales en unidades orquestables.

In [ ]:
from prefect import flow, task
from prefect.logging import get_run_logger
import pandas as pd
import os, tempfile, json

# Crear datos de ejemplo si no existen
DATA_DIR = "02-ejemplos/pipeline_completo/data"
os.makedirs(f"{DATA_DIR}/raw", exist_ok=True)
os.makedirs(f"{DATA_DIR}/processed", exist_ok=True)

if not os.path.exists(f"{DATA_DIR}/raw/tickets.csv"):
    import numpy as np
    np.random.seed(42)
    categorias = ['red', 'system', 'database', 'security', 'application']
    descripciones = [
        'El servidor de producción no responde',
        'Error de conexión a la base de datos PostgreSQL',
        'Ataque de fuerza bruta detectado en el firewall',
        'La aplicación web tarda más de 30 segundos en cargar',
        'El certificado SSL ha expirado',
        'No se puede acceder al share de red',
        'El servicio de correo no envía notificaciones',
        'La CPU del servidor principal está al 95%',
        'Error 500 en el endpoint de pagos',
        'El backup nocturno no se ha completado',
    ] * 20
    
    data = []
    for desc in descripciones:
        cat = np.random.choice(categorias)
        data.append({'description': desc, 'category': cat, 'priority': np.random.choice(['alta', 'media', 'baja'])})
    
    df = pd.DataFrame(data)
    df.to_csv(f"{DATA_DIR}/raw/tickets.csv", index=False)
    print(f'✅ Datos de ejemplo creados: {len(df)} registros')
else:
    df = pd.read_csv(f"{DATA_DIR}/raw/tickets.csv")
    print(f'✅ Datos existentes: {len(df)} registros')

### 2.1 Definir las tareas del pipeline

Cada función decorada con `@task` es una unidad de trabajo independiente.
Prefect resuelve las dependencias automáticamente según los argumentos.

In [ ]:
@task
def load_data(path: str) -> pd.DataFrame:
    """Carga datos desde un archivo CSV."""
    logger = get_run_logger()
    logger.info(f"Cargando datos desde {path}")
    
    df = pd.read_csv(path)
    logger.info(f"Registros cargados: {len(df)}")
    return df


@task
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia y prepara los datos."""
    logger = get_run_logger()
    logger.info("Iniciando limpieza de datos")
    
    # Eliminar nulos en columnas críticas
    before = len(df)
    df = df.dropna(subset=['description', 'category'])
    
    # Normalizar texto
    df['description'] = df['description'].str.lower().str.strip()
    
    logger.info(f"Registros tras limpieza: {len(df)} (eliminados {before - len(df)})")
    return df


@task
def train_model(df: pd.DataFrame, max_features: int = 5000) -> str:
    """Entrena un modelo y registra en MLflow."""
    logger = get_run_logger()
    logger.info(f"Iniciando entrenamiento con max_features={max_features}")
    
    import mlflow
    from sklearn.model_selection import train_test_split
    
    # Preparar datos
    vectorizer = TfidfVectorizer(max_features=max_features)
    X = vectorizer.fit_transform(df['description'])
    y = df['category']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    with mlflow.start_run():
        # Entrenar
        model = LogisticRegression(max_iter=1000, random_state=42)
        model.fit(X_train, y_train)
        
        # Evaluar
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')
        
        # Registrar
        mlflow.log_param("max_features", max_features)
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_param("data_records", len(df))
        mlflow.sklearn.log_model(model, "model")
        
        run_id = mlflow.active_run().info.run_id
        logger.info(f"Modelo entrenado. Accuracy={acc:.3f}, F1={f1:.3f}")
        logger.info(f"MLflow Run ID: {run_id}")
        
        # Registrar en Model Registry
        mlflow.register_model(
            f"runs:/{run_id}/model",
            "TicketClassifier"
        )
        logger.info("Modelo registrado en Model Registry")
    
    return run_id

### 2.2 Ensamblar el flujo

El `@flow` define el orden de ejecución. Prefect se encarga de:
- Ejecutar las tareas en el orden correcto (resolviendo dependencias)
- Pasar los resultados de una tarea a la siguiente
- Registrar logs, duración y estado de cada tarea

In [ ]:
@flow(name="Pipeline Entrenamiento ML")
def pipeline_entrenamiento(data_path: str, max_features: int = 5000):
    """Orquesta el pipeline completo: datos → entrenamiento → registro."""
    logger = get_run_logger()
    logger.info("="*50)
    logger.info("🚀 INICIANDO PIPELINE DE ENTRENAMIENTO")
    logger.info("="*50)
    
    # Tarea 1: Cargar datos
    df = load_data(data_path)
    
    # Tarea 2: Limpiar datos
    df_clean = clean_data(df)
    
    # Tarea 3: Entrenar modelo
    run_id = train_model(df_clean, max_features=max_features)
    
    logger.info("="*50)
    logger.info(f"✅ PIPELINE COMPLETADO. Run ID: {run_id}")
    logger.info("="*50)
    
    return run_id

### 2.3 Ejecutar el flujo

In [ ]:
# Ejecutar el pipeline
run_id = pipeline_entrenamiento(
    data_path=f"{DATA_DIR}/raw/tickets.csv",
    max_features=5000
)
print(f"\n✅ Pipeline ejecutado. MLflow Run ID: {run_id}")

---
## Parte 3: Reintentos y Recuperación ante Fallos

Una de las ventajas clave de Prefect es la recuperación automática.
Vamos a crear una tarea que falla intermitentemente y configurar reintentos.

In [ ]:
import random
from prefect import task
from datetime import timedelta


@task(
    retries=3,
    retry_delay_seconds=5,
    retry_delay_threshold=timedelta(minutes=2),
    name="Cargar desde API externa"
)
def load_from_api(source: str) -> dict:
    """Simula una carga desde API externa que puede fallar."""
    logger = get_run_logger()
    logger.info(f"⏳ Intentando cargar desde {source}...")
    
    if random.random() < 0.6:  # 60% de probabilidad de fallo
        logger.warning(f"⚠️ Fallo en la conexión con {source}, reintentando...")
        raise ConnectionError(f"Timeout conectando a {source}")
    
    logger.info(f"✅ Datos cargados desde {source}")
    return {"source": source, "status": "ok", "records": 150}


@flow(name="Pipeline con Reintentos")
def pipeline_resiliente():
    """Flujo que demuestra reintentos automáticos."""
    logger = get_run_logger()
    
    try:
        result = load_from_api("https://api.tickets.example.com/v1/incidents")
        logger.info(f"Resultado final: {result}")
    except Exception as e:
        logger.error(f"❌ Pipeline falló tras reintentos: {e}")
        logger.info("🔔 Notificación: Se requiere intervención manual")


# Ejecutar varias veces para ver reintentos en acción
for i in range(3):
    print(f"\n--- Ejecución {i+1} ---")
    pipeline_resiliente()

**Pregunta**: ¿Qué observas en las ejecuciones? ¿Cómo afectan los reintentos
al resultado final del pipeline? ¿Qué implicación tiene esto para RA4a
(automatización de decisiones)?

*Escribe tu respuesta aquí*

---
## Parte 4: Pipeline Multi-Tarea con Dependencias

Ahora vamos a crear un flujo más complejo que refleje el pipeline real
de producción, con dependencias entre tareas.

In [ ]:
@task
def validate_data(df: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema de los datos."""
    logger = get_run_logger()
    
    try:
        import pandera as pa
        schema = pa.DataFrameSchema({
            "description": pa.Column(str, pa.Check.str_length(1, 10000), nullable=False),
            "category": pa.Column(str, pa.Check.isin(["red", "system", "database", "security", "application"]), nullable=False),
            "priority": pa.Column(str, pa.Check.isin(["alta", "media", "baja"]), nullable=True),
        })
        schema.validate(df)
        logger.info("✅ Validación de datos superada")
    except ImportError:
        logger.warning("⚠️ Pandera no disponible, validación manual")
        assert df['description'].notna().all(), "Hay descripciones nulas"
        logger.info("✅ Validación manual superada")
    
    return df


@task
def save_processed_data(df: pd.DataFrame, output_path: str) -> str:
    """Guarda los datos procesados."""
    logger = get_run_logger()
    df.to_csv(output_path, index=False)
    logger.info(f"✅ Datos guardados en {output_path}")
    return output_path


@task
def notify_completion(run_id: str, data_path: str):
    """Simula una notificación de finalización."""
    logger = get_run_logger()
    logger.info(f"📧 Notificación: Pipeline completado")
    logger.info(f"   Run ID: {run_id}")
    logger.info(f"   Datos: {data_path}")
    logger.info(f"   Estado: Completed ✅")


@flow(name="Pipeline Completo ML")
def pipeline_completo():
    """Pipeline completo con validación, guardado y notificación."""
    logger = get_run_logger()
    
    data_path = f"{DATA_DIR}/raw/tickets.csv"
    output_path = f"{DATA_DIR}/processed/tickets_clean.csv"
    
    # Paso 1: Cargar
    df = load_data(data_path)
    
    # Paso 2: Limpiar
    df_clean = clean_data(df)
    
    # Paso 3: Validar
    df_validated = validate_data(df_clean)
    
    # Paso 4: Guardar datos procesados
    saved_path = save_processed_data(df_validated, output_path)
    
    # Paso 5: Entrenar
    run_id = train_model(df_validated)
    
    # Paso 6: Notificar
    notify_completion(run_id, saved_path)
    
    return run_id


run_id = pipeline_completo()
print(f"\n✅ Pipeline completo ejecutado con éxito. Run ID: {run_id}")

---
## Parte 5: Conexión con el Stack Convergente

El pipeline que acabas de crear conecta directamente con:

| Componente | Cómo se conecta |
|------------|----------------|
| **F2 Pipeline datos** | Las tareas `load_data` y `clean_data` replican el pipeline ETL |
| **F3 MLflow** | La tarea `train_model` registra experimentos en MLflow y promueve al Model Registry |
| **F5 FastAPI** | El modelo registrado será servido por la API en la práctica P5 |
| **F7 Observabilidad** | Prefect registra duración y estado de cada tarea - datos para dashboards |

**RA4a en acción**: Este pipeline automatiza decisiones que antes requerían
intervención manual: ¿cuándo re-entrenar? → Prefect decide basándose en el
calendario. ¿Qué hacer si falla? → reintentos automáticos. ¿Quién lo notifica?
→ tarea de notificación integrada.

---
## Parte 6: Ejercicios

### Ejercicio 1: Pipeline con parámetros configurables

Modifica el `pipeline_completo` para aceptar parámetros que permitan
configurar el tipo de modelo (`LogisticRegression`, `RandomForestClassifier`)
y el número de `max_features` para el vectorizador.

In [ ]:
# TODO: Crea un pipeline_configurable que acepte model_type y max_features
# Pista: define el modelo como parámetro y usa if/elif para seleccionar

@flow(name="Pipeline Configurable")
def pipeline_configurable(
    data_path: str = f"{DATA_DIR}/raw/tickets.csv",
    model_type: str = "LogisticRegression",
    max_features: int = 5000
):
    # Tu código aquí
    pass

### Ejercicio 2: Notificación Slack/Email

Implementa una tarea que envíe una notificación simulada (print) cuando el
pipeline falla después de todos los reintentos. Incluye información sobre
qué tarea falló y cuántos reintentos se intentaron.

In [ ]:
# TODO: Implementa notificación de fallo

@task
def notify_failure(task_name: str, attempts: int, error: str):
    # Tu código aquí
    pass

### Ejercicio 3: Análisis RA4a

Responde: ¿Qué decisiones del pipeline ML se han automatizado gracias a
Prefect? ¿Qué decisiones siguen requiriendo intervención humana?
¿Cómo mejorarías este pipeline para alinearlo con RA4a?

In [ ]:
# Escribe tu análisis aquí

---
## Resumen de la Práctica

✅ Has creado un pipeline ML orquestado con Prefect
✅ Has implementado reintentos y recuperación ante fallos
✅ Has conectado Prefect con MLflow para registro automático
✅ Has construido un flujo multi-tarea con dependencias
✅ Has analizado cómo la orquestación automatiza decisiones (RA4a)

**Próxima práctica (P5)**: Crearás una API FastAPI para servir el modelo
que Prefect ha entrenado y registrado en MLflow.